# Sentiment Classification with Word Embeddings (spaCy)

In this notebook we go through **word embeddings** using `spaCy`, and use them as features to build a text classifier.
This is our assignment notebook: we build a **sentiment classifier** for app reviews, and compare two ways of representing
the text numerically: **spaCy word embeddings** and **TF-IDF**. Both representations will be evaluated using the
**same classifier and the same preprocessing pipeline**, so the comparison is fair.

Update pip tools and install spacy

`pip install -U pip setuptools wheel`

`pip install -U spacy`

Download the English medium model (it ships with real, pretrained word vectors, unlike the small model):

`python -m spacy download en_core_web_md`

In [2]:
!pip install -U spacy


#

In [3]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 57.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import spacy
import numpy as np
import pandas as pd
import re

nlp = spacy.load('en_core_web_md')

# Vectors!

The vectors generated by the `spacy` model are 300-dimensional, pretrained on a large English corpus.
Let's check how similar some words are to each other, using their vectors.

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

words = ['good', 'great', 'bad', 'terrible', 'app']
vectors = np.array([nlp(w).vector for w in words])
similarities = cosine_similarity(vectors, vectors)

pd.DataFrame(similarities, index=words, columns=words).round(2)

,good,great,bad,terrible,app
good,1.00,0.52,1.00,0.74,0.10
great,0.52,1.00,0.52,0.38,0.11
bad,1.00,0.52,1.00,0.74,0.10
terrible,0.74,0.38,0.74,1.00,0.05
app,0.10,0.11,0.10,0.05,1.00


As expected, `good` and `great` are much closer to each other than `good` and `bad`, which shows the model captured some notion of meaning.

In [7]:
vector = nlp("Amazing").vector
print(f"Vector shape: {vector.shape}")
vector[:10]

Vector shape: (300,)


array([-0.76677  ,  0.52154  ,  0.51221  ,  0.17084  ,  0.022783 ,
        0.084228 ,  0.24177  ,  0.34026  ,  0.0044662,  1.5543   ],
      dtype=float32)

## Load the dataset

We'll use a dataset of app reviews with 3 sentiment classes: `0 = negative`, `1 = neutral`, `2 = positive`.

In [9]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('all_data.csv')
df = df.dropna(subset=['review', 'sentiment']).reset_index(drop=True)
print(df.shape)
df['sentiment'].value_counts()

Saving all_data.csv to all_data.csv
(40516, 2)


,count
sentiment,
2,15302
0,13466
1,11748


## Basic Preprocessing

Same simple, lowercase + non-alphabetic cleanup preprocessing is applied **before** building either representation,
this keeps the comparison fair since both models will see the exact same cleaned text.

In [10]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_review'] = df['review'].map(preprocess_text)
df = df[df['clean_review'].str.len() > 0].reset_index(drop=True)
print(df.shape)
df[['review', 'clean_review']].head()

(39804, 3)


,review,clean_review
0,Aditya Ingole Deaf,aditya ingole deaf
1,I love the app.! There is no issue but if u co...,i love the app there is no issue but if u coul...
2,"So hard to use. The web app failed, and the mo...",so hard to use the web app failed and the mobi...
3,I hate that the app makes a sound every time s...,i hate that the app makes a sound every time s...
4,Useless at BSE star MF meet.voice too mych slo...,useless at bse star mf meet voice too mych slo...


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 31843, Test size: 7961


## Embeddings as feature

We can use the average word embedding of a sentence as its feature vector, and build a classifier using them.

In [12]:
def get_avg_vectors(texts, nlp_model):
    return np.array([doc.vector for doc in nlp_model.pipe(texts, batch_size=256)])

X_train_spacy = get_avg_vectors(X_train, nlp)
X_test_spacy = get_avg_vectors(X_test, nlp)
print(X_train_spacy.shape, X_test_spacy.shape)

(31843, 300) (7961, 300)


# Train a classifier

We'll use a `LinearSVC` classifier, it's simple, fast and works well as a baseline for text classification.

In [13]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, f1_score

clf_spacy = LinearSVC(random_state=42, max_iter=5000)
clf_spacy.fit(X_train_spacy, y_train)

pred_spacy = clf_spacy.predict(X_test_spacy)
acc_spacy = accuracy_score(y_test, pred_spacy)
f1_spacy = f1_score(y_test, pred_spacy, average='macro')

print(f"spaCy embeddings -> Accuracy: {acc_spacy:.4f} | Macro F1: {f1_spacy:.4f}")
print(classification_report(y_test, pred_spacy))

spaCy embeddings -> Accuracy: 0.5434 | Macro F1: 0.4953
              precision    recall  f1-score   support

           0       0.54      0.63      0.58      2657
           1       0.45      0.18      0.26      2302
           2       0.57      0.74      0.65      3002

    accuracy                           0.54      7961
   macro avg       0.52      0.52      0.50      7961
weighted avg       0.52      0.54      0.51      7961



## Comparing with TF-IDF

The assignment asks us to compare the spaCy embeddings representation above with a **TF-IDF** representation,
using the **same model** (`LinearSVC`) and the **same preprocessed text**, so the only thing that changes is
how the text gets turned into numbers.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print(X_train_tfidf.shape, X_test_tfidf.shape)

(31843, 10000) (7961, 10000)


In [15]:
clf_tfidf = LinearSVC(random_state=42, max_iter=5000)
clf_tfidf.fit(X_train_tfidf, y_train)

pred_tfidf = clf_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, pred_tfidf)
f1_tfidf = f1_score(y_test, pred_tfidf, average='macro')

print(f"TF-IDF -> Accuracy: {acc_tfidf:.4f} | Macro F1: {f1_tfidf:.4f}")
print(classification_report(y_test, pred_tfidf))

TF-IDF -> Accuracy: 0.6543 | Macro F1: 0.6354
              precision    recall  f1-score   support

           0       0.69      0.69      0.69      2657
           1       0.54      0.43      0.48      2302
           2       0.68      0.80      0.74      3002

    accuracy                           0.65      7961
   macro avg       0.64      0.64      0.64      7961
weighted avg       0.65      0.65      0.65      7961



# Get top similar

In [16]:
print(f"spaCy embeddings : accuracy = {acc_spacy:.4f}, macro F1 = {f1_spacy:.4f}")
print(f"TF-IDF          : accuracy = {acc_tfidf:.4f}, macro F1 = {f1_tfidf:.4f}")

spaCy embeddings : accuracy = 0.5434, macro F1 = 0.4953
TF-IDF          : accuracy = 0.6543, macro F1 = 0.6354


In [17]:
import random

def predict_sentiment(text, vectorizer_fn, clf):
    clean = preprocess_text(text)
    vec = vectorizer_fn([clean])
    label = clf.predict(vec)[0]
    mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}
    return mapping[label]

sample = "the app keeps crashing every time i try to join a call"
print("spaCy prediction :", predict_sentiment(sample, lambda t: get_avg_vectors(t, nlp), clf_spacy))
print("TF-IDF prediction:", predict_sentiment(sample, tfidf.transform, clf_tfidf))

spaCy prediction : negative
TF-IDF prediction: negative


# Conclusion

- Word embeddings are a very powerful feature, especially with small/noisy data, since the model reuses meaning
  it already learned from a huge external corpus instead of learning everything from scratch.
- On this specific dataset (short, noisy, multilingual app reviews), **TF-IDF slightly outperformed the spaCy
  embeddings** in our experiment. This makes sense: a lot of the vocabulary here is slang/typos/short phrases
  that the general-purpose pretrained vectors don't represent well, while TF-IDF can directly pick up on the exact
  words (like "crash", "good", "useless") that are strongly correlated with sentiment in this domain.
- Both representations were evaluated with the exact same classifier (`LinearSVC`) and the exact same
  preprocessing pipeline, so the comparison reflects a real difference between the representations, not a
  difference caused by the model or the cleaning steps.